# Synthetic ground-truth validation, calibrated to realistic colocalization magnitude

Companion to `synthetic_ground_truth_colocalization.ipynb`. That notebook's ground truth
(60 shared + 40/40 exclusive puncta in a 256x256x15 volume) produces a true odds ratio of
~293 and a raw (no deconvolution) odds ratio of ~32 -- both far higher than what this project's
own real data shows (real `MB_syn`/`MB_non_syn` odds ratios are ~1.7-2.1, per
`coloc_results.csv`). That mismatch isn't a display issue -- it means the puncta density/
overlap-fraction parameters there don't represent this project's actual biological regime.

This notebook re-derives the ground-truth puncta density so that **raw** (undeconvolved)
odds ratio lands near the real ~2, at **SNR matching the real range**
(`snr_min_HSP_Mito` ~6-35, median ~10-14, from `coloc_results.csv`) -- not a
lower/noisier SNR chosen to force a more dramatic-looking recovery curve. Under this
calibration, at realistic SNR, the recovery curve from raw to iteration 7 was found (during
the design process for this notebook, see chat history) to be **nearly flat** -- closely
matching the ~1.09x raw-vs-deconvolved effect size already found on real data. That is not a
failure of the simulation -- once calibrated to a realistic starting magnitude, the model
reproduces the modest real effect size without being tuned to do so, which is itself a form
of independent validation. Pushing SNR below the real range does produce a more dramatic
curve, but at that point the simulation is illustrating the deconvolution mechanism in the
abstract, not this project's actual acquisitions -- so it isn't used here as the headline
result (see the SNR-sweep section for that comparison, shown explicitly rather than hidden).

Same optics as the companion notebook: real `2026_06_15_Carmina` acquisition, scene 0, HSP
(Cy5, ch1) vs Mito (Alexa546, ch2), ~1.0 AU pinhole. Same metric definitions (fixed
95th-percentile threshold, full 3D stack -- matching `compute_colocalization` exactly, see
section 4) and same two deconvolution algorithms (blind vs. classical Richardson-Lucy).

In [ ]:
%load_ext autoreload
%autoreload 2

import platform
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.stats import fisher_exact, pearsonr
from skimage.restoration import richardson_lucy

system = platform.system()
if system == 'Linux':
    home = '/home/gerard/'
elif system == 'Darwin':
    home = '/Users/gerard/'
elif system == "Windows":
    home = 'C:/Users/cviko/'

try:
    sys.path.append(os.path.abspath(os.path.join(os.pardir, 'src')))
    from data_processing import (parse_lif_psf_params, describe_acquisition,
                                  _gaussian_psf, _blind_rl, _pinhole_psf_factor)
except ImportError:
    path2add = home + 'analysis/confocal/src'
    sys.path.append(path2add)
    from data_processing import (parse_lif_psf_params, describe_acquisition,
                                  _gaussian_psf, _blind_rl, _pinhole_psf_factor)

home = home + 'data/confocal/'


## 1. Optics: read real PSF parameters from the 2026_06_15_Carmina acquisition

Same NA / refractive index / voxel size / pinhole / emission wavelengths the real pipeline
uses for this session -- read from `.lif` metadata rather than hardcoded, per project
convention.

In [ ]:
lif_path = home + '2026_06_15_Carmina/Project.lif'
scene = 0

params = parse_lif_psf_params(lif_path, scene)
NA = params['NA']
n = params['n']
vxy = params['voxel_xy_um']
vz = params['voxel_z_um']

info = describe_acquisition(lif_path, do_print=False)
pinhole_au = info[list(info.keys())[scene]]['sequences'][0]['pinhole_airy']
pinhole_factor = _pinhole_psf_factor(pinhole_au)

# ch1 = HSP (Cy5), ch2 = Mito (Alexa546) -- the primary colocalization pair
CH_A = {'name': 'HSP (Cy5)', 'emission_nm': params['emission_nm'][1]}
CH_B = {'name': 'Mito (Alexa546)', 'emission_nm': params['emission_nm'][2]}

def sigma_px(emission_nm):
    lam = emission_nm * 1e-3  # nm -> um
    sigma_xy_um = 0.21 * lam / NA
    sigma_z_um = 0.66 * lam * n / (NA ** 2)
    return sigma_xy_um / vxy, sigma_z_um / vz

for ch in (CH_A, CH_B):
    sxy, sz = sigma_px(ch['emission_nm'])
    ch['sigma_xy_px'] = sxy * pinhole_factor
    ch['sigma_z_px'] = sz * pinhole_factor
    print(f"{ch['name']}: em={ch['emission_nm']:.0f} nm  "
          f"sigma_xy={ch['sigma_xy_px']:.2f} px  sigma_z={ch['sigma_z_px']:.2f} px")

print(f'NA={NA}, n={n}, voxel_xy={vxy:.4f} um/px, voxel_z={vz:.4f} um/step, '
      f'pinhole={pinhole_au:.3f} AU (factor={pinhole_factor:.3f})')


## 2. Ground-truth generator

Places small point-like "puncta" (much smaller than the PSF, so post-blur size is
PSF-dominated -- same regime as real sub-resolution BRP/HSP/Mito puncta) into two 3D
(Z, Y, X) channel volumes. Each punctum site is one of three categories:

- **shared** -- present in both channel A and channel B at the same location (true positive overlap)
- **A only** / **B only** -- present in one channel only (true negative overlap)

The fraction of shared vs exclusive sites directly controls the true colocalization strength,
so the ground truth is known exactly, before any blur or noise is added.

In [ ]:
rng = np.random.default_rng(0)

def _add_blob(vol, center, sigma, amplitude):
    z0, y0, x0 = center
    r = max(2, int(round(3 * sigma)))
    zlo, zhi = max(0, z0 - r), min(vol.shape[0], z0 + r + 1)
    ylo, yhi = max(0, y0 - r), min(vol.shape[1], y0 + r + 1)
    xlo, xhi = max(0, x0 - r), min(vol.shape[2], x0 + r + 1)
    zz, yy, xx = np.mgrid[zlo:zhi, ylo:yhi, xlo:xhi]
    blob = amplitude * np.exp(-((zz - z0)**2 + (yy - y0)**2 + (xx - x0)**2) / (2 * sigma**2))
    vol[zlo:zhi, ylo:yhi, xlo:xhi] += blob


def _sample_positions(n, shape, margin, min_dist, rng, existing=()):
    # margin: (margin_z, margin_y, margin_x) -- Z is typically much shallower than
    # Y/X (few tens of frames vs hundreds of pixels), so it needs its own, smaller
    # margin rather than one shared value that could exceed shape[0]//2.
    mz, my, mx = margin
    positions = list(existing)
    new_positions = []
    attempts = 0
    while len(new_positions) < n and attempts < n * 300:
        z = rng.integers(mz, shape[0] - mz)
        y = rng.integers(my, shape[1] - my)
        x = rng.integers(mx, shape[2] - mx)
        if all((z - pz)**2 + (y - py)**2 + (x - px)**2 >= min_dist**2
               for pz, py, px in positions + new_positions):
            new_positions.append((z, y, x))
        attempts += 1
    if len(new_positions) < n:
        print(f'Warning: only placed {len(new_positions)}/{n} puncta (volume too crowded '
              f'for min_dist={min_dist}); increase volume size or lower min_dist.')
    return new_positions


def generate_ground_truth(shape, n_shared, n_A_only, n_B_only, punctum_sigma=0.8,
                           min_dist=10, margin=(3, 10, 10), intensity_range=(0.6, 1.0), rng=rng):
    vol_A = np.zeros(shape, dtype=np.float64)
    vol_B = np.zeros(shape, dtype=np.float64)

    shared_pos = _sample_positions(n_shared, shape, margin, min_dist, rng)
    A_only_pos = _sample_positions(n_A_only, shape, margin, min_dist, rng, existing=shared_pos)
    B_only_pos = _sample_positions(n_B_only, shape, margin, min_dist, rng,
                                    existing=shared_pos + A_only_pos)

    for pos in shared_pos:
        _add_blob(vol_A, pos, punctum_sigma, rng.uniform(*intensity_range))
        _add_blob(vol_B, pos, punctum_sigma, rng.uniform(*intensity_range))
    for pos in A_only_pos:
        _add_blob(vol_A, pos, punctum_sigma, rng.uniform(*intensity_range))
    for pos in B_only_pos:
        _add_blob(vol_B, pos, punctum_sigma, rng.uniform(*intensity_range))

    sites = {'shared': shared_pos, 'A_only': A_only_pos, 'B_only': B_only_pos}
    return vol_A, vol_B, sites


## 3. Forward imaging model

Blur each channel with the real (session-specific) anisotropic PSF sigma, then add
Poisson shot noise plus a small amount of Gaussian read noise on top of a background
offset, matching the noise composition real confocal data has. Defaults
(`peak_counts=220, background=22, read_noise_std=3.5`) were tuned by trial so the
synthetic data's `snr_proxy` lands at ~13 -- the median of the real `2026_06_15`/
`2026_06_25` sessions' `snr_min_HSP_Mito` (range ~6-35, median ~10-14).

`blurred_only` skips the noise step entirely -- PSF blur with no acquisition noise on top.
It answers "how much does noise alone cost us", **not** "what is the best deconvolution can
do" -- deconvolution's whole job is to undo blur, so a working deconvolution is *expected*
to score above this line, not capped by it. (An earlier draft of this notebook treated
`blurred_only` as a ceiling and read iterations exceeding it as a noise-amplification red
flag -- section 9 below shows that reading was wrong: deconvolving the *noise-free* blurred
data crosses the same line just as fast, so the crossing is diffraction-blur removal working
as intended, not a noise artifact. Kept here only as "cost of noise alone", each recovered
point should instead be judged against the pre-blur truth line.)

In [ ]:
def simulate_acquisition(vol, sigma_xy_px, sigma_z_px, peak_counts=220, background=22,
                          read_noise_std=3.5, rng=rng, add_noise=True):
    blurred = gaussian_filter(vol, sigma=(sigma_z_px, sigma_xy_px, sigma_xy_px))
    if not add_noise:
        return blurred
    scale = blurred.max()
    scaled = (blurred / scale * peak_counts if scale > 0 else blurred) + background
    noisy = rng.poisson(scaled).astype(np.float64)
    noisy += rng.normal(0, read_noise_std, size=noisy.shape)
    return np.clip(noisy, 0, None).astype(np.uint16)


## 4. Colocalization metrics

Matches `compute_colocalization`/`thresholding` in `src/colocalization.py`/`data_processing.py`
exactly (see the correction note above): each channel is thresholded at its own **fixed
95th percentile** (`np.percentile(img, 95)`, keeps exactly the top 5% of voxels), computed on
the **full 3D stack**, not a 2D projection -- not Otsu, and not the CLAUDE.md "Option A"
write-up's sum-projection suggestion, since that isn't what the production code actually runs.

Otherwise: Manders-style directional fractions, Fisher-exact odds ratio (`alternative='greater'`),
chance-corrected enrichment (`observed/expected`), and Pearson r on continuous intensities within
the union foreground mask. Applied identically to the ground truth and to every processing
stage, so the numbers are directly comparable.

In [ ]:
def coloc_metrics(imgA, imgB, percentile=95):
    imgA = np.asarray(imgA, dtype=np.float64)
    imgB = np.asarray(imgB, dtype=np.float64)

    thrA = np.percentile(imgA, percentile)
    thrB = np.percentile(imgB, percentile)
    maskA = imgA > thrA
    maskB = imgB > thrB

    N = imgA.size
    A_and_B = int((maskA & maskB).sum())
    A_not_B = int((maskA & ~maskB).sum())
    B_not_A = int((~maskA & maskB).sum())
    neither = N - A_and_B - A_not_B - B_not_A

    odds_ratio, _ = fisher_exact([[A_and_B, A_not_B], [B_not_A, neither]], alternative='greater')

    pA = maskA.sum() / N
    pB = maskB.sum() / N
    expected = pA * pB * N
    enrichment = A_and_B / expected if expected > 0 else np.nan

    fraction_A_in_B = A_and_B / maskA.sum() if maskA.sum() > 0 else np.nan
    fraction_B_in_A = A_and_B / maskB.sum() if maskB.sum() > 0 else np.nan

    union_mask = maskA | maskB
    if union_mask.sum() > 1:
        r, _ = pearsonr(imgA[union_mask].ravel(), imgB[union_mask].ravel())
    else:
        r = np.nan

    return {
        'odds_ratio': odds_ratio, 'enrichment': enrichment,
        # log-transformed: matches how the real pipeline treats OR everywhere (pooled_results.ipynb's
        # log_MB_*_odds_ratio columns) -- OR is a multiplicative ratio with a null at 1 (not 0), so
        # it's right-skewed on a linear scale; log makes over- and under-estimation symmetric (2x too
        # high and 2x too low land equally far from 0). enrichment is left linear -- the real analysis
        # doesn't log it, so this notebook doesn't either, even though the same argument would apply.
        'log_odds_ratio': np.log(odds_ratio),
        'fraction_A_in_B': fraction_A_in_B, 'fraction_B_in_A': fraction_B_in_A,
        'pearson_r': r,
    }


## 5. Deconvolution algorithms: blind vs. classical Richardson-Lucy, 2D-per-frame

Two algorithms, both applied frame-by-frame (this dataset's sigma_z >= 2 px so the automatic
Nyquist check would otherwise pick 3D mode -- matching the `forced2d=True` production
setting used for blind RL):

- **`blind_richardson_lucy`** (production choice, `num_iter=7`): `_blind_rl`, same code as
  `deconvolve()` -- alternately refines the image estimate *and* the PSF itself, seeded from
  the theoretical Gaussian PSF rather than trusting it as fixed.
- **`richardson_lucy`** (classical/plain RL, `deconvolve()`'s other default path): skimage's
  `richardson_lucy`, run against a **fixed** theoretical Gaussian PSF that is never updated --
  the only difference from blind RL is that the PSF never adapts, so any PSF-size mismatch
  (a common issue when no bead calibration is available) directly propagates into the result.

In [ ]:
def deconvolve_stack_blind(stack, sigma_xy_px, num_iter):
    psf_init = _gaussian_psf(sigma_xy_px)
    out = np.empty(stack.shape, dtype=np.float32)
    for z in range(stack.shape[0]):
        img = stack[z].astype(np.float64)
        scale = img.max()
        img_n = img / scale if scale > 0 else img
        est, _ = _blind_rl(img_n, psf_init, num_iter)
        out[z] = (est * scale).astype(np.float32)
    return out


def deconvolve_stack_classical(stack, sigma_xy_px, num_iter):
    psf = _gaussian_psf(sigma_xy_px)
    out = np.empty(stack.shape, dtype=np.float32)
    for z in range(stack.shape[0]):
        img = stack[z].astype(np.float64)
        scale = img.max()
        img_n = img / scale if scale > 0 else img
        est = richardson_lucy(img_n, psf, num_iter=num_iter, clip=True)
        out[z] = (est * scale).astype(np.float32)
    return out


ALGORITHMS = {
    'blind_richardson_lucy': deconvolve_stack_blind,
    'richardson_lucy': deconvolve_stack_classical,
}


## 6. Run: generate data, sweep iteration count, compare to ground truth

Volume: 15 Z-frames x 256 x 256 XY. Puncta density reduced from the companion notebook's
60 shared + 40/40 exclusive down to **12 shared + 92/92 exclusive** -- found by trial (see
intro) to put raw (undeconvolved) odds ratio near this project's real ~2, instead of ~32.

`min_dist` (minimum spacing enforced between *any* two puncta at placement, including two
*different* non-shared ones) is raised from the companion notebook's 5 px to **10 px**, since
5 px is barely larger than sigma_xy (~4.45 px here) -- close enough that unrelated (not truly
shared) puncta placed near each other by chance can still have their post-blur footprints
overlap, creating spurious "colocalization" that has nothing to do with the true label. That
artifact was directly visible in an earlier version of this notebook (see chat history): the
noise-free `blurred_only` fraction came out *higher* than the true fraction, which should be
impossible if blur only ever loses information relative to the pre-blur point sources.
At `min_dist=10`, `blurred_only` correctly falls below true, as expected.

Noise stays at the same `simulate_acquisition` defaults as the companion notebook
(`peak_counts=220, background=22, read_noise_std=3.5`), which were themselves tuned to match
the real `snr_min_HSP_Mito` median (~13) -- i.e. nothing here is tuned to make the recovery
curve look any particular way, only to match real magnitude and real noise level. All metrics
computed on the full 3D stack (matches `compute_colocalization`'s actual behavior, see
section 4).

In [ ]:
shape = (15, 256, 256)
vol_A, vol_B, sites = generate_ground_truth(
    shape, n_shared=12, n_A_only=92, n_B_only=92,
    punctum_sigma=0.8, min_dist=10, margin=(3, 10, 10),
)
print({k: len(v) for k, v in sites.items()})

true_metrics = coloc_metrics(vol_A, vol_B)
true_metrics


In [ ]:
noisy_A = simulate_acquisition(vol_A, CH_A['sigma_xy_px'], CH_A['sigma_z_px'])
noisy_B = simulate_acquisition(vol_B, CH_B['sigma_xy_px'], CH_B['sigma_z_px'])

from colocalization import snr_proxy
print('snr_proxy A (HSP) :', snr_proxy(noisy_A))
print('snr_proxy B (Mito):', snr_proxy(noisy_B))

blurred_only_A = simulate_acquisition(vol_A, CH_A['sigma_xy_px'], CH_A['sigma_z_px'], add_noise=False)
blurred_only_B = simulate_acquisition(vol_B, CH_B['sigma_xy_px'], CH_B['sigma_z_px'], add_noise=False)
blurred_only_metrics = coloc_metrics(blurred_only_A, blurred_only_B)
blurred_only_metrics


> **Known limitation: this calibration matches real odds ratio/enrichment, not real fraction.**
> `n_shared=12` was chosen so raw odds ratio lands near this project's real ~2 (see above).
> That leaves `fraction_A_in_B`/`fraction_B_in_A` far below their real counterparts (~0.08
> here vs. ~0.2-0.3 in `pooled_results.ipynb`'s real `fraction of HSP in Mito` columns) --
> because with a fixed 95th-percentile mask, fraction is essentially `n_shared / (n_shared +
> n_A_only)`, a direct ratio of puncta counts.
>
> This was tested deliberately, not left unexamined: holding `n_shared=12` fixed and reducing
> `n_A_only` to push the shared:total ratio up to match the real fraction (~0.23, at
> `n_A_only=40`) drives raw odds ratio to 6.35 -- over 3x too high. The two metrics can't be
> matched simultaneously by tuning puncta counts in this model, because both are driven by the
> same underlying quantity here (how much of the fixed-size mask is genuine punctum signal vs.
> background padding). That is itself a real finding, not just an unresolved parameter search:
> it suggests real HSP/Mito spatial colocalization isn't well described by "some puncta
> perfectly co-located, the rest fully independent" -- a graded/probabilistic overlap model, or
> background-level intensity correlation between channels, would be needed to match both
> metrics at once, and hasn't been attempted here.
>
> Practical consequence: treat this notebook's `log_odds_ratio`/`enrichment` recovery curves as
> the calibrated, real-magnitude result. Treat its `fraction_A_in_B`/`fraction_B_in_A` curves as
> showing the correct *qualitative* behavior (direction, whether deconvolution helps) but at an
> uncalibrated, too-low *absolute* magnitude relative to real data.

In [ ]:
iterations_to_test = [0, 2, 4, 7, 10, 15]  # 0 = raw, no deconvolution

rows = []
for algo_name, algo_fn in ALGORITHMS.items():
    for it in iterations_to_test:
        if it == 0:
            procA = noisy_A.astype(np.float64)
            procB = noisy_B.astype(np.float64)
        else:
            procA = algo_fn(noisy_A, CH_A['sigma_xy_px'], it)
            procB = algo_fn(noisy_B, CH_B['sigma_xy_px'], it)
        m = coloc_metrics(procA, procB)
        m['num_iter'] = it
        m['algorithm'] = algo_name
        rows.append(m)
        print(f"{algo_name:22s} iter={it:2d}  odds_ratio={m['odds_ratio']:.3f}  "
              f"enrichment={m['enrichment']:.3f}  pearson_r={m['pearson_r']:.3f}")

df = pd.DataFrame(rows)
df


## 7. Recovery vs. iteration count

In [ ]:
metrics_to_plot = ['log_odds_ratio', 'enrichment', 'fraction_A_in_B', 'fraction_B_in_A', 'pearson_r']
algo_colors = {'blind_richardson_lucy': 'tab:orange', 'richardson_lucy': 'tab:green'}

for metric in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(6, 4))
    for algo_name in ALGORITHMS:
        sub = df[df.algorithm == algo_name].sort_values('num_iter')
        ax.plot(sub['num_iter'], sub[metric], 'o-', color=algo_colors[algo_name], label=algo_name)
    ax.axhline(true_metrics[metric], color='k', linestyle='--', label='true (pre-blur)')
    ax.axhline(blurred_only_metrics[metric], color='tab:blue', linestyle='-.',
               label='diffraction limit (blur, no noise)')
    ax.axvline(7, color='r', linestyle=':', alpha=0.5, label='production (iter=7)')
    ax.set_xlabel('RL iterations')
    ax.set_title(metric)
    ax.legend(fontsize=7)
    fig.tight_layout()


## 7b. Same recovery curves, relative to the true (known) value

The absolute plots above have to fit the true (pre-blur) reference line, which sits far above
where any recovered curve reaches -- so the recovered curves get compressed into a thin band
near the bottom and their own iteration-to-iteration shape is hard to read. Re-expressing each
curve relative to the *true* value fixes that AND directly answers the more relevant question
("how close is this to the actual known answer"), rather than the answer to "how much better
than doing nothing" the earlier raw-relative version gave.

- **`log_odds_ratio`**: plotted as `log(value) - log(true)` = `log(value/true)`. 0 = exact match
  to true; negative = underestimate. Odds ratio is a multiplicative ratio with a null at 1
  (not 0, see section 4/section 7), so this -- not a plain `value/true` ratio -- is the correct
  way to express "how far from true," for the same reason the metric itself is logged in the
  first place.
- **`enrichment` / `fraction_A_in_B` / `fraction_B_in_A`**: plotted as the plain ratio
  `value/true` (1.0 = exact match to true). Enrichment has the same null-at-1 structure as
  odds ratio, so the same log argument would technically apply -- but it's kept linear here to
  match the real analysis, which doesn't log enrichment either.

Unlike the raw-relative version, true is now directly representable *on* the plot (it's just
the reference line at 0 or 1) rather than being off-scale text -- because raw and the
diffraction limit are both below true here, not above it. `pearson_r` is still excluded
(same reason as before: not a ratio-like quantity, and can be negative).

In [ ]:
log_relative_metrics = ['log_odds_ratio']
ratio_relative_metrics = ['enrichment', 'fraction_A_in_B', 'fraction_B_in_A']

for metric in log_relative_metrics + ratio_relative_metrics:
    is_log = metric in log_relative_metrics
    raw_value = df.loc[df.num_iter == 0, metric].iloc[0]  # same for both algorithms

    def rel(x):
        return (x - true_metrics[metric]) if is_log else (x / true_metrics[metric])

    raw_rel = rel(raw_value)
    blur_rel = rel(blurred_only_metrics[metric])
    true_rel = 0.0 if is_log else 1.0

    fig, ax = plt.subplots(figsize=(6, 4))
    for algo_name in ALGORITHMS:
        sub = df[df.algorithm == algo_name].sort_values('num_iter')
        ax.plot(sub['num_iter'], rel(sub[metric]), 'o-', color=algo_colors[algo_name], label=algo_name)
    ax.axhline(true_rel, color='k', linestyle='--', label='true (pre-blur)')
    ax.axhline(raw_rel, color='gray', linewidth=0.8, linestyle=':', label='raw')
    ax.axhline(blur_rel, color='tab:blue', linestyle='-.', label='diffraction limit (blur, no noise)')
    ax.axvline(7, color='r', linestyle=':', alpha=0.5, label='production (iter=7)')
    ax.set_xlabel('RL iterations')
    ax.set_ylabel('log(value/true)' if is_log else 'value / true')
    ax.set_title(metric + ('  (log relative to true)' if is_log else '  (relative to true)'))
    ax.legend(fontsize=7)
    fig.tight_layout()


## 8. Control: does deconvolving noise-free data show the same rise?

If the noise-free curve rises through the `blurred_only` line just as readily as the noisy
curve, that proves the rise is deconvolution genuinely undoing blur (toward the true value),
not noise being sharpened into spurious structure. If instead the noisy curve rose
*faster/higher* than the noise-free curve, that would indicate a real noise-amplification
artifact inflating the metric beyond genuine recovery.

In [ ]:
rows_nonoise = []
for algo_name, algo_fn in ALGORITHMS.items():
    for it in iterations_to_test:
        if it == 0:
            pA, pB = blurred_only_A, blurred_only_B
        else:
            pA = algo_fn(blurred_only_A, CH_A['sigma_xy_px'], it)
            pB = algo_fn(blurred_only_B, CH_B['sigma_xy_px'], it)
        m = coloc_metrics(pA, pB)
        m['num_iter'] = it
        m['algorithm'] = algo_name
        rows_nonoise.append(m)

df_nonoise = pd.DataFrame(rows_nonoise)

for metric in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(6, 4))
    for algo_name in ALGORITHMS:
        sub_noise = df[df.algorithm == algo_name].sort_values('num_iter')
        sub_nonoise = df_nonoise[df_nonoise.algorithm == algo_name].sort_values('num_iter')
        ax.plot(sub_noise['num_iter'], sub_noise[metric], 'o-',
                color=algo_colors[algo_name], label=f'{algo_name} (with noise)')
        ax.plot(sub_nonoise['num_iter'], sub_nonoise[metric], 's--',
                color=algo_colors[algo_name], alpha=0.5, label=f'{algo_name} (no noise)')
    ax.axhline(true_metrics[metric], color='k', linestyle='--', alpha=0.6, label='true (pre-blur)')
    ax.axvline(7, color='r', linestyle=':', alpha=0.5, label='production (iter=7)')
    ax.set_xlabel('RL iterations')
    ax.set_title(metric)
    ax.legend(fontsize=7)
    fig.tight_layout()


## 9. Visual check: raw vs. deconvolved vs. ground truth (single mid-stack frame)

In [ ]:
z_mid = shape[0] // 2
iter_to_show = 7

decA_blind = deconvolve_stack_blind(noisy_A, CH_A['sigma_xy_px'], iter_to_show)
decB_blind = deconvolve_stack_blind(noisy_B, CH_B['sigma_xy_px'], iter_to_show)
decA_classical = deconvolve_stack_classical(noisy_A, CH_A['sigma_xy_px'], iter_to_show)
decB_classical = deconvolve_stack_classical(noisy_B, CH_B['sigma_xy_px'], iter_to_show)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (vol_true, noisy, dec_blind, dec_classical, name) in enumerate([
    (vol_A, noisy_A, decA_blind, decA_classical, CH_A['name']),
    (vol_B, noisy_B, decB_blind, decB_classical, CH_B['name']),
]):
    axes[row, 0].imshow(vol_true[z_mid], cmap='gray')
    axes[row, 0].set_title(f'{name}: ground truth')
    axes[row, 1].imshow(noisy[z_mid], cmap='gray')
    axes[row, 1].set_title(f'{name}: raw (blurred + noise)')
    axes[row, 2].imshow(dec_blind[z_mid], cmap='gray')
    axes[row, 2].set_title(f'{name}: blind RL (iter={iter_to_show})')
    axes[row, 3].imshow(dec_classical[z_mid], cmap='gray')
    axes[row, 3].set_title(f'{name}: classical RL (iter={iter_to_show})')
    for ax in axes[row]:
        ax.axis('off')
fig.tight_layout()


## 10. Robustness check: does this hold across SNR, puncta density, and algorithm?

Section 7's recovery curve is one specific synthetic condition (one density, one SNR level,
one random seed). Check whether its shape (flat vs. rising, and by how much) is consistent
across a grid of SNR values spanning the real range (~8/13/25, matching `snr_min_HSP_Mito`),
puncta density around the section 6 calibration (sparse/moderate/dense), and **both
algorithms** -- blind vs. classical (fixed-PSF) RL. All three density conditions are scaled
around the section 6 calibration (12 shared / 92 exclusive each), not around the companion
notebook's much denser, unrealistic-magnitude setup.

Only `num_iter` in `[0, 7, 15]` is tested per condition (raw, production, and a high-iteration
check) to keep the grid tractable -- this is about the *direction and consistency* of the
effect across conditions, not another full iteration sweep. 3 density x 3 SNR x 2 algorithms
x 3 iterations = 54 deconvolution runs.

In [ ]:
snr_conditions = {
    'low_snr':    dict(peak_counts=150, background=25, read_noise_std=4.0),   # snr_proxy ~ 6-8
    'medium_snr': dict(peak_counts=220, background=22, read_noise_std=3.5),   # snr_proxy ~ 13 (same as section 6)
    'high_snr':   dict(peak_counts=300, background=20, read_noise_std=3.0),   # snr_proxy ~ 20-25
}
density_conditions = {
    'sparse':   dict(n_shared=6,  n_A_only=46,  n_B_only=46),
    'moderate': dict(n_shared=12, n_A_only=92,  n_B_only=92),    # same as section 6
    'dense':    dict(n_shared=24, n_A_only=184, n_B_only=184),
}

sweep_rows = []
for density_name, density_params in density_conditions.items():
    vA, vB, _ = generate_ground_truth(shape, punctum_sigma=0.8, min_dist=10,
                                       margin=(3, 10, 10), **density_params)
    tm = coloc_metrics(vA, vB)
    for snr_name, snr_params in snr_conditions.items():
        nA = simulate_acquisition(vA, CH_A['sigma_xy_px'], CH_A['sigma_z_px'], **snr_params)
        nB = simulate_acquisition(vB, CH_B['sigma_xy_px'], CH_B['sigma_z_px'], **snr_params)
        for algo_name, algo_fn in ALGORITHMS.items():
            for it in [0, 7, 15]:
                if it == 0:
                    pA, pB = nA.astype(np.float64), nB.astype(np.float64)
                else:
                    pA = algo_fn(nA, CH_A['sigma_xy_px'], it)
                    pB = algo_fn(nB, CH_B['sigma_xy_px'], it)
                m = coloc_metrics(pA, pB)
                m.update(density=density_name, snr=snr_name, algorithm=algo_name, num_iter=it,
                          true_fraction_A_in_B=tm['fraction_A_in_B'],
                          true_odds_ratio=tm['odds_ratio'])
                sweep_rows.append(m)
        print(f'done: density={density_name}  snr={snr_name}')

df_sweep = pd.DataFrame(sweep_rows)
df_sweep


Summarize each condition as the *relative* change from raw (iter=0) to iter=15 -- this
directly tests "does deconvolution move this metric much, relative to where raw already is,"
independent of each condition's very different absolute scale. Raw (iter=0) doesn't depend
on algorithm, so it's shared between the two algorithm rows for the same density/SNR
condition -- only the iter=15 endpoint differs.

In [ ]:
def relative_change(df_sweep, metric):
    rows = []
    for (density, snr, algorithm), g in df_sweep.groupby(['density', 'snr', 'algorithm']):
        raw_val = g.loc[g.num_iter == 0, metric].iloc[0]
        iter15_val = g.loc[g.num_iter == 15, metric].iloc[0]
        rel = (iter15_val - raw_val) / raw_val if raw_val != 0 else np.nan
        rows.append({'density': density, 'snr': snr, 'algorithm': algorithm,
                      'metric': metric, 'relative_change': rel})
    return pd.DataFrame(rows)

metrics_check = ['fraction_A_in_B', 'fraction_B_in_A', 'odds_ratio', 'enrichment']
rel_df = pd.concat([relative_change(df_sweep, m) for m in metrics_check], ignore_index=True)
pivot = rel_df.pivot_table(index=['density', 'snr', 'algorithm'], columns='metric', values='relative_change')
pivot = pivot[metrics_check]
pivot


In [ ]:
for algo_name in ALGORITHMS:
    sub_pivot = pivot.xs(algo_name, level='algorithm')
    x = np.arange(len(sub_pivot))
    width = 0.2

    fig, ax = plt.subplots(figsize=(9, 5))
    for i, metric in enumerate(metrics_check):
        ax.bar(x + i * width, sub_pivot[metric].values, width, label=metric)
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels([f'{d}\n{s}' for d, s in sub_pivot.index], fontsize=8)
    ax.axhline(0, color='k', linewidth=0.8)
    ax.set_ylabel('relative change, raw -> iter=15')
    ax.set_title(f'Deconvolution effect size by metric, across SNR x puncta-density -- {algo_name}')
    ax.legend(fontsize=8)
    fig.tight_layout()
